In [ ]:
# 1. 安裝必要套件
!pip install -U sentence-transformers jieba pandas numpy

import pandas as pd
import numpy as np
import jieba
import re
from collections import Counter
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer

# ==========================================
# 第一部分：配置與模組化函數
# ==========================================

def get_embedding_model(model_name='paraphrase-multilingual-MiniLM-L12-v2'):
    """
    初始化 Embedding 模型。
    參考 method1 的多國語言配置。
    """
    print(f"正在載入模型: {model_name}...")
    return SentenceTransformer(model_name)

def clean_text(text):
    """清理文字，僅保留中文字符"""
    return re.sub(r'[^\u4e00-\u9fa5]', '', str(text))

def get_word_frequencies(corpus):
    """對語料庫進行斷詞並統計詞頻"""
    words = []
    for text in corpus:
        words.extend(jieba.lcut(clean_text(text)))
    return Counter([w for w in words if len(w) > 1]) # 排除單字

def calculate_log_odds_ratio(target_counts, ref_counts):
    """
    實作 Log-odds Ratio 演算法。
    用於找出在目標桶中相對於基準桶更顯著的特徵詞。
    """
    all_words = set(list(target_counts.keys()) + list(ref_counts.keys()))
    n_target = sum(target_counts.values())
    n_ref = sum(ref_counts.values())
    
    log_odds = {}
    for word in all_words:
        # 加上平滑項 (Prior) 避免除以零
        p_i = (target_counts[word] + 0.1) / (n_target + 0.1)
        q_i = (ref_counts[word] + 0.1) / (n_ref + 0.1)
        # Log-odds 核心公式
        log_odds[word] = np.log(p_i / (1 - p_i)) - np.log(q_i / (1 - q_i))
    
    return sorted(log_odds.items(), key=lambda x: x[1], reverse=True)

def validate_by_projection(word, current_bad_center, model, threshold=0.45):
    """
    向量投影/相似度校驗。
    確保新詞在語意空間中與目前的惡意概念中心足夠接近。
    """
    word_vec = model.encode([word])
    sim = cosine_similarity(word_vec, current_bad_center.reshape(1, -1))[0][0]
    return sim >= threshold, sim

# ==========================================
# 第二部分：主流程實作
# ==========================================

def run_bootstrapping_pipeline(df_corpus, initial_seeds, n_target=20):
    """
    執行完整的關鍵詞擴增迭代
    """
    model = get_embedding_model()
    target_keywords = set(initial_seeds)
    all_texts = df_corpus.copy()
    
    # 預先計算全語料 Embedding 以加速搜尋
    print("正在計算全語料 Embedding...")
    all_embeddings = model.encode(all_texts.tolist(), show_progress_bar=True)
    
    iteration = 0
    while len(target_keywords) < n_target:
        iteration += 1
        print(f"\n>>> 迭代 {iteration} | 目前關鍵詞數: {len(target_keywords)}")
        
        # 步驟 1 & 2: 建立壞桶與好桶 (基準桶)
        pattern = '|'.join([re.escape(k) for k in target_keywords])
        bad_mask = all_texts.str.contains(pattern, na=False)
        bad_bucket = all_texts[bad_mask]
        good_bucket = all_texts[~bad_mask] # 修正後的「好桶」定義為一般背景語料
        
        if bad_bucket.empty:
            print("警告：壞桶為空，請檢查初始種子。")
            break

        # 步驟 3: 語意擴張 (透過 BERT 找出相似內容形成大壞桶)
        bad_vecs = model.encode(bad_bucket.tolist())
        bad_center = bad_vecs.mean(axis=0)
        
        # 尋找與壞桶中心最相似的前 50 篇文章
        sims = cosine_similarity(bad_center.reshape(1, -1), all_embeddings)[0]
        big_bad_indices = sims.argsort()[-50:][::-1]
        big_bad_bucket = all_texts.iloc[big_bad_indices]
        
        # 步驟 4: 計算 Log-odds 找出特徵詞
        target_counts = get_word_frequencies(big_bad_bucket)
        ref_counts = get_word_frequencies(good_bucket.sample(min(2000, len(good_bucket))))
        
        candidate_words = calculate_log_odds_ratio(target_counts, ref_counts)
        
        # 步驟 5: 校驗並更新關鍵詞集
        added_count = 0
        for word, score in candidate_words:
            if word not in target_keywords:
                # 向量投影校驗：防止語意偏移 (Semantic Drift)
                is_valid, sim_score = validate_by_projection(word, bad_center, model)
                if is_valid:
                    target_keywords.add(word)
                    added_count += 1
                    print(f"  [新增] {word} (Log-odds: {score:.2f}, Sim: {sim_score:.2f})")
            
            if len(target_keywords) >= n_target or added_count >= 5:
                break
        
        if added_count == 0:
            print("無法再找到符合條件的新詞，停止擴增。")
            break
            
    return list(target_keywords)

# ==========================================
# 第三部分：資料載入與執行
# ==========================================

# 讀取你的 Series 資料 (此處範例假設讀取 CSV 後轉 Series)
# fb_data = pd.read_csv("your_data.csv")['content'] 

# 模擬測試數據
test_data = pd.Series([
    "博弈娛樂城，儲值就送點數", "專業下注教學，百家樂必勝", "今日早餐吃蛋餅好滿足", 
    "投資詐騙手法大公開", "這家珊瑚色唇膏好漂亮", "運動彩券即時比分", 
    "地下錢莊高利貸追債", "線上麻將現金交易", "台北美食餐廳推薦"
])

seeds = ["下注", "賭博", "娛樂城"]
final_keywords = run_bootstrapping_pipeline(test_data, seeds, n_target=20)

# 儲存結果
output_df = pd.DataFrame(final_keywords, columns=["keyword"])
output_df.to_csv("output.csv", index=False, encoding="utf-8-sig")
print("\n任務完成！結果已儲存至 output.csv")